
#### DDL & DML Operations

  - CREATE / DROP / USE databases
  - Managed vs External tables
  - CTAS (Create Table As Select)
  - INSERT INTO / INSERT OVERWRITE
  - ALTER TABLE, DESCRIBE, SHOW commands
  - Table properties and partitioning via SQL


##### DDL/DML Summary

In [0]:
"""
──────────────────────────────────────────────────────────────────────
MANAGED TABLE                      EXTERNAL TABLE
──────────────────────────────────────────────────────────────────────
Spark owns data + metadata         Spark owns metadata only
DROP TABLE → data deleted          DROP TABLE → data preserved
Default warehouse location         Custom LOCATION required
──────────────────────────────────────────────────────────────────────

CTAS:
  - Schema inferred from SELECT (cannot pre-declare)
  - Data populated immediately
  - CREATE TABLE ... AS SELECT ...

INSERT INTO      → append rows
INSERT OVERWRITE → replace all data (non-Delta) or atomic replace (Delta)

DESCRIBE <table>           → column names + types
DESCRIBE EXTENDED <table>  → + table type, location, properties
DESCRIBE DETAIL <table>    → Delta-specific file/version statistics

"""


##### DATABASE operations(CREATE/DROP/USE)

In [0]:
from pyspark.sql import SparkSession
import os

# ---------------------------------------------------------------------------
# 1. DATABASE operations
# ---------------------------------------------------------------------------
spark.sql("DROP DATABASE IF EXISTS retail CASCADE")
spark.sql("CREATE DATABASE retail COMMENT 'Retail data warehouse'")
spark.sql("USE retail")

print("=== 1. Show databases ===")
spark.sql("SHOW DATABASES").show()
spark.sql("DESCRIBE DATABASE retail").show(truncate=False)

##### MANAGED TABLE (default)
 - Spark owns both metadata AND data files
 - DROP TABLE → removes data from warehouse dir
 - Location: spark.sql.warehouse.dir/db_name/table_name

In [0]:
# ---------------------------------------------------------------------------
# 2. MANAGED TABLE (default)
#    - Spark owns both metadata AND data files
#    - DROP TABLE → removes data from warehouse dir
#    - Location: spark.sql.warehouse.dir/<db>/<table>
# ---------------------------------------------------------------------------
spark.sql("DROP TABLE IF EXISTS retail.orders")
spark.sql("""
    CREATE TABLE retail.orders (
        order_id    INT,
        customer_id INT,
        product     STRING,
        quantity    INT,
        unit_price  DOUBLE,
        order_date  DATE,
        status      STRING
    )
    USING delta
    COMMENT 'Managed table — Spark controls data lifecycle'
""")

print("=== 2. Managed table created ===")
spark.sql("DESCRIBE EXTENDED retail.orders").show(truncate=False)


##### EXTERNAL TABLE
- Spark owns metadata ONLY; data lives at a custom LOCATION
- DROP TABLE → removes metadata only, data files remain
- Key exam distinction vs managed tables

In [0]:

# ---------------------------------------------------------------------------
# 3. EXTERNAL TABLE
#    - Spark owns metadata ONLY; data lives at a custom LOCATION
#    - DROP TABLE → removes metadata only, data files remain
#    - Key exam distinction vs managed tables
# ---------------------------------------------------------------------------
ext_path = "/tmp/external_customers"
spark.sql("DROP TABLE IF EXISTS retail.customers_ext")
spark.sql(f"""
    CREATE TABLE retail.customers_ext (
        customer_id INT,
        name        STRING,
        email       STRING,
        city        STRING
    )
    USING parquet
    LOCATION '{ext_path}'
    COMMENT 'External table — data survives DROP TABLE'
""")

print("=== 3. External table — note Type=EXTERNAL ===")
spark.sql("DESCRIBE EXTENDED retail.customers_ext") \
     .filter("col_name IN ('Type','Location','Provider')") \
     .show(truncate=False)


##### CTAS — Create Table As Select
- Schema is INFERRED from the SELECT result (cannot specify schema)
- Always creates a MANAGED table unless LOCATION is specified
- Populates data immediately

In [0]:

# ---------------------------------------------------------------------------
# 4. CTAS — Create Table As Select
#    - Schema is INFERRED from the SELECT result (cannot specify schema)
#    - Always creates a MANAGED table unless LOCATION is specified
#    - Populates data immediately
# ---------------------------------------------------------------------------
raw_data = [(1, 101, "Laptop",  2, 999.99,  "2024-01-10", "shipped"),
            (2, 102, "Mouse",   5, 29.99,   "2024-01-11", "delivered"),
            (3, 101, "Keyboard",3, 79.99,   "2024-01-12", "pending"),
            (4, 103, "Monitor", 1, 399.99,  "2024-01-13", "shipped"),
            (5, 104, "Laptop",  1, 999.99,  "2024-01-14", "delivered"),
            (6, 102, "Headset", 2, 149.99,  "2024-01-15", "pending")]

raw_df = spark.createDataFrame(
    raw_data,
    "order_id INT, customer_id INT, product STRING, quantity INT, unit_price DOUBLE, order_date STRING, status STRING"
)
raw_df.createOrReplaceTempView("raw_orders")

spark.sql("DROP TABLE IF EXISTS retail.shipped_orders")
spark.sql("""
    CREATE TABLE retail.shipped_orders
    USING delta
    AS
    SELECT
        order_id,
        customer_id,
        product,
        quantity,
        ROUND(quantity * unit_price, 2) AS total_value,
        order_date,
        status
    FROM raw_orders
    WHERE status IN ('shipped', 'delivered')
""")

print("=== 4. CTAS result ===")
spark.sql("SELECT * FROM retail.shipped_orders").show()


##### INSERT INTO  vs  INSERT OVERWRITE

In [0]:

# ---------------------------------------------------------------------------
# 5. INSERT INTO  vs  INSERT OVERWRITE
#    INSERT INTO      → appends rows (additive)
#    INSERT OVERWRITE → replaces ALL data in table (or partition)
# ---------------------------------------------------------------------------
spark.sql("DROP TABLE IF EXISTS retail.order_log")
spark.sql("""
    CREATE TABLE retail.order_log (
        order_id   INT,
        product    STRING,
        status     STRING
    ) USING delta
""")

# First load
spark.sql("""
    INSERT INTO retail.order_log
    VALUES (1, 'Laptop', 'shipped'), (2, 'Mouse', 'delivered')
""")
print("=== 5a. After INSERT INTO ===")
spark.sql("SELECT * FROM retail.order_log").show()

# Second load — appends
spark.sql("""
    INSERT INTO retail.order_log
    VALUES (3, 'Keyboard', 'pending')
""")
print("=== 5b. After 2nd INSERT INTO (appended) ===")
spark.sql("SELECT * FROM retail.order_log").show()

# Overwrite — replaces all
spark.sql("""
    INSERT OVERWRITE retail.order_log
    VALUES (99, 'Monitor', 'new')
""")
print("=== 5c. After INSERT OVERWRITE (all previous rows gone) ===")
spark.sql("SELECT * FROM retail.order_log").show()


##### Alter table/Describe/Show commands

In [0]:

# ---------------------------------------------------------------------------
# 6. ALTER TABLE
# ---------------------------------------------------------------------------
# spark.sql("ALTER TABLE retail.order_log SET TBLPROPERTIES ('owner' = 'data_team')")
spark.sql("ALTER TABLE retail.order_log RENAME TO retail.order_log_v2")

print("=== 6. Table after rename ===")
spark.sql("SHOW TABLES IN retail").show()


In [0]:
# ---------------------------------------------------------------------------
# 7. DESCRIBE commands — all variants
# ---------------------------------------------------------------------------
print("=== 7a. DESCRIBE (column info) ===")
spark.sql("DESCRIBE retail.shipped_orders").show()

print("=== 7b. DESCRIBE EXTENDED (full metadata) ===")
spark.sql("DESCRIBE EXTENDED retail.shipped_orders").show(50, truncate=False)

print("=== 7c. DESCRIBE DETAIL (Delta specific) ===")
try:
    spark.sql("DESCRIBE DETAIL retail.shipped_orders").show(truncate=False)
except Exception:
    print("DESCRIBE DETAIL is Delta Lake / Databricks-specific")


In [0]:
# ---------------------------------------------------------------------------
# 8. SHOW commands
# ---------------------------------------------------------------------------
print("=== 8. SHOW commands ===")
spark.sql("SHOW TABLES IN retail").show()
spark.sql("SHOW COLUMNS IN retail.shipped_orders").show()
spark.sql("SHOW CREATE TABLE retail.shipped_orders").show(truncate=False)
spark.sql("SHOW TBLPROPERTIES retail.order_log_v2").show(truncate=False)

##### Table properties and partitioning via SQL

In [0]:
# ---------------------------------------------------------------------------
# 9. Partitioned Table via SQL
#    Partitioning physically organizes data into sub-directories
# ---------------------------------------------------------------------------
spark.sql("DROP TABLE IF EXISTS retail.orders_partitioned")
spark.sql("""
    CREATE TABLE retail.orders_partitioned (
        order_id   INT,
        product    STRING,
        quantity   INT,
        unit_price DOUBLE
    )
    USING DELTA
    PARTITIONED BY (status STRING, order_date STRING)
""")

spark.sql("""
    INSERT INTO retail.orders_partitioned
    PARTITION (status='shipped', order_date='2024-01-10')
    VALUES (1, 'Laptop', 2, 999.99)
""")
spark.sql("""
    INSERT INTO retail.orders_partitioned
    PARTITION (status='pending', order_date='2024-01-11')
    VALUES (2, 'Mouse', 5, 29.99)
""")

print("=== 9. Partitioned table data ===")
spark.sql("SELECT * FROM retail.orders_partitioned").show()
spark.sql("SHOW PARTITIONS retail.orders_partitioned").show()

spark.stop()
